# Evidence-Grounded, Alignment-Aware Fine-Grained Image Difference Understanding

Given **Image A** and **Image B**, this notebook establishes *what* changed, *where* (boxes and masks in **both original images**), *how* the state differs, *which evidence supports or contradicts* each conclusion, and *when the evidence is insufficient*. It runs on a single 16 GB GPU (e.g. RTX A4000) and only uses pretrained models.

**Principle.** Every model output is an *observation* with provenance. Claims are accepted only by an explicit, correlation-aware policy, never by majority vote. Final language is produced **after** verification, and it is checked against the accepted claims.

```mermaid
flowchart TD
  IN[Image A + Image B + scope] --> A[Stage A: load, EXIF, quality, DINOv2 compat]
  A --> B[Stage B: LightGlue / LoFTR / ORB<br/>domain-aware model selection<br/>alignment quality + local uncertainty]
  B --> C[Stage C: tolerant Lab / edge / texture maps<br/>contrast-conditional robust z<br/>pre- vs post-alignment residual]
  B --> E[Stage E: OCR + line matching<br/>edit ops + re-read + pixel check]
  B --> F[Stage F: AnyChange masks, experimental]
  A --> D1[Stage D: M3Diff captions<br/>own env, subprocess]
  A --> D2[Stage D: Qwen3-VL hypotheses]
  C & E & F & D1 & D2 --> G0[Candidate groups, merged with provenance]
  G0 --> R[Iterative re-inspection:<br/>native-resolution pixels + neutral VLM crop check<br/>+ grounding of free-text hypotheses]
  R --> G[Stage G: evidence graph<br/>families: appearance / ocr / vlm / m3diff / learned]
  G --> V{Claim verifier}
  V -->|accepted| S[Structured claims JSON]
  V -->|needs_review| S
  V -->|abstained| S
  S --> H[Stage H: calibration, if a labelled calibrator exists]
  H --> I[Stage I: controlled verbalizer<br/>validated; template fallback]
  I --> OUT[Report + overlays + maps + masks + JSON]
```

Coordinates are `[x1, y1, x2, y2]` in **original pixels after EXIF orientation**, with the origin at top-left and x2/y2 exclusive. `M_ba` maps B→A, and A is the reference frame.

**Sections:** 1 setup & install · 2 environment · 3 model inventory · 4 configuration · 5 component self-test · 6 inputs · 7 run · 8 inspect results (validation, alignment, maps, candidates, OCR, semantics, evidence graph, claims, overlays, JSON) · 9 export · 10 evaluation & ablations · 11 calibration · 12 troubleshooting · 13 cleanup

## 1. Setup and modular installation
Set the flags and run the cell. Each group installs separately, so a failing optional stack never breaks the core. Nothing is reinstalled if it is already present. **Install PyTorch with CUDA first** (see `requirements-core.txt`).

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("project root:", ROOT)

INSTALL_CORE      = False   # numpy/opencv/transformers/... (requirements-core.txt)
INSTALL_MATCHING  = False   # kornia + LightGlue
INSTALL_OCR       = False   # easyocr
INSTALL_4BIT      = False   # bitsandbytes (only for vlm_4bit=True)
INSTALL_ANYCHANGE = False   # torchange (experimental)
PREFETCH_MODELS   = False   # download pretrained weights into the HF cache once (no re-download later)

def pip(*args):
    print("$ pip", " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args], capture_output=True, text=True)
    print("   ->", "OK" if r.returncode == 0 else "FAILED\n" + r.stderr[-2000:])
    return r.returncode == 0

def missing(mod): return importlib.util.find_spec(mod) is None

if INSTALL_CORE:      pip("-r", "requirements-core.txt")
if INSTALL_MATCHING:
    if missing("kornia"):    pip("kornia>=0.7.3")
    if missing("lightglue"): pip("git+https://github.com/cvg/LightGlue.git")
if INSTALL_OCR and missing("easyocr"):        pip("easyocr==1.7.2")
if INSTALL_4BIT and missing("bitsandbytes"):  pip("bitsandbytes>=0.45")
if INSTALL_ANYCHANGE and missing("torchange"): pip("torchange")

In [ ]:
# Optional: prefetch weights once (cached in HF_HOME / ~/.cache/huggingface; later runs load from cache)
if PREFETCH_MODELS:
    from huggingface_hub import snapshot_download
    for rid in ["facebook/dinov2-small", "Qwen/Qwen3-VL-4B-Instruct"]:
        try: print(rid, "->", snapshot_download(rid))
        except Exception as e: print(rid, "FAILED:", e)
    ck = ROOT / "checkpoints" / "sam_vit_h_4b8939.pth"     # only needed for AnyChange
    if INSTALL_ANYCHANGE and not ck.exists():
        import urllib.request; ck.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth", ck)

## 2. Environment report
Records Python, OS, GPU and VRAM, CUDA build, library versions, disk, RAM, network reachability, and whether an HF token is present. The report is saved with every result.

In [ ]:
import json, pandas as pd
from idu.env import environment_report, MODEL_INVENTORY
ENV = environment_report()
libs = ENV.pop("libraries")
display(pd.DataFrame([ENV]).T.rename(columns={0: "value"}))
display(pd.DataFrame([libs]).T.rename(columns={0: "version"}))
json.dump({**ENV, "libraries": libs}, open(ROOT / "environment_report.json", "w"), indent=2)
if not ENV.get("cuda_available"):
    print("WARNING: no CUDA GPU visible - the classical preset works on CPU; VLM stages will be very slow.")

## 3. Model inventory
The upstream repository, paper, checkpoint and license were verified on 2026-10-12. `status` refers to *upstream verification*. Whether a component is installed and working on **this** machine is shown by the self-test in §4.

> **License note:** SuperPoint weights are **non-commercial**. Use `lightglue_features='disk'` or `'aliked'` for commercial work.

In [ ]:
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(MODEL_INVENTORY)[["component", "name", "status", "checkpoint", "license", "hardware", "repo", "limitations"]])

## 4. Configuration
Choose a preset, then adjust options. Every option is saved in the result JSON.

| preset | runs |
|---|---|
| `classical` | ORB alignment + low-level evidence (no downloads, CPU-capable) |
| `ocr` | classical + LightGlue + DINOv2 + OCR |
| `semantic` | classical + LightGlue + DINOv2 + Qwen3-VL |
| `full` | everything enabled below (M3Diff/AnyChange are opt-in) |

In [ ]:
from idu import make_config, Pipeline, PRESETS
try:
    import ipywidgets as W
    w = dict(
        preset=W.Dropdown(options=list(PRESETS), value="full", description="preset"),
        domain=W.Dropdown(options=["auto", "document", "screenshot", "product", "scene", "fixed_camera", "nonrigid"], value="auto", description="domain"),
        scope=W.Text(value="", placeholder="optional, e.g. 'only the label text'", description="scope"),
        matcher=W.Dropdown(options=["lightglue", "loftr", "orb"], value="lightglue", description="matcher"),
        lightglue_features=W.Dropdown(options=["superpoint", "disk", "aliked"], value="superpoint", description="LG features"),
        use_ocr=W.Checkbox(value=True, description="OCR"),
        use_vlm=W.Checkbox(value=True, description="VLM (Qwen3-VL)"),
        vlm_model=W.Text(value="Qwen/Qwen3-VL-4B-Instruct", description="VLM id"),
        vlm_4bit=W.Checkbox(value=False, description="VLM 4-bit"),
        use_m3diff=W.Checkbox(value=False, description="M3Diff (own env)"),
        m3diff_python=W.Text(value="", placeholder="/path/to/envs/m3diff/bin/python", description="m3diff py"),
        use_anychange=W.Checkbox(value=False, description="AnyChange"),
        use_reinspection=W.Checkbox(value=True, description="re-inspection"),
        ocr_langs=W.Text(value="en", description="OCR langs"),
    )
    display(W.VBox([W.HBox([w["preset"], w["domain"], w["matcher"], w["lightglue_features"]]), w["scope"],
                    W.HBox([w["use_ocr"], w["use_vlm"], w["vlm_4bit"], w["use_reinspection"]]), w["vlm_model"],
                    W.HBox([w["use_m3diff"], w["m3diff_python"]]), W.HBox([w["use_anychange"], w["ocr_langs"]])]))
except ImportError:
    w = None
    print("ipywidgets not installed - edit MANUAL_CFG in the next cell instead.")

In [ ]:
MANUAL_CFG = dict(preset="full")   # used only without ipywidgets

def current_config():
    if w is None:
        kw = dict(MANUAL_CFG); return make_config(kw.pop("preset"), **kw)
    v = {k: x.value for k, x in w.items()}
    preset = v.pop("preset")
    v["ocr_langs"] = [s.strip() for s in v["ocr_langs"].split(",") if s.strip()]
    cfg = make_config(preset)
    TOGGLES = ("use_ocr", "use_vlm", "use_m3diff", "use_anychange", "matcher")   # owned by the preset unless 'full'
    for k, val in v.items():
        if k in TOGGLES and preset != "full":
            continue
        setattr(cfg, k, val)
    return cfg

CFG = current_config()
print({k: getattr(CFG, k) for k in ["preset", "domain", "matcher", "use_ocr", "use_vlm", "vlm_model", "vlm_4bit", "use_m3diff", "use_anychange"]})

## 5. Component self-test (installation status on this machine)
This cell loads each enabled specialist and runs **one minimal inference** on a synthetic pair. A row is marked `tested` only if that inference returned. Failed rows include the fix. Heavy models are released afterwards.

In [ ]:
from idu.pipeline import component_self_test
CFG = current_config()
ST = component_self_test(CFG, run_m3diff=False)    # set run_m3diff=True to also run one M3Diff inference
display(pd.DataFrame(ST))

## 6. Inputs: upload two unseen images (or give paths)
PNG, JPEG, WebP, TIFF, BMP and other formats are accepted. EXIF orientation is corrected and the originals are kept. If nothing is provided, a synthetic demo pair with known changes is used.

In [ ]:
PATH_A, PATH_B = "", ""          # optional file paths
up_a = up_b = None
try:
    import ipywidgets as W
    up_a = W.FileUpload(accept="image/*", multiple=False, description="Image A")
    up_b = W.FileUpload(accept="image/*", multiple=False, description="Image B")
    display(W.HBox([up_a, up_b]))
except ImportError:
    pass
IN_COLAB = "google.colab" in sys.modules
print("In Colab, if the widget upload does not respond, set USE_COLAB_UPLOAD=True in the next cell.")

In [ ]:
USE_COLAB_UPLOAD = False

def _upload_bytes(u):
    if u is None or not u.value: return None
    v = u.value
    item = v[0] if isinstance(v, (tuple, list)) else next(iter(v.values()))   # ipywidgets 8 / 7
    return bytes(item["content"])

src_a = _upload_bytes(up_a) or (PATH_A or None)
src_b = _upload_bytes(up_b) or (PATH_B or None)
if USE_COLAB_UPLOAD and IN_COLAB:
    from google.colab import files
    print("upload Image A"); fa = files.upload(); src_a = next(iter(fa.values()))
    print("upload Image B"); fb = files.upload(); src_b = next(iter(fb.values()))
DEMO_GT = None
if src_a is None or src_b is None:
    from idu.evaluate import make_pair
    src_a, src_b, DEMO_GT = make_pair("text_edit+color_change+shift", 42)
    print("No inputs given -> synthetic demo pair. Ground truth:", DEMO_GT)

## 7. Run the pipeline
Re-run this cell after changing the config or the inputs. Loaded models are cached across runs, so nothing is re-downloaded.

In [ ]:
CFG = current_config()
try:
    PIPE.reconfigure(CFG)
except NameError:
    PIPE = Pipeline(CFG)
RES, ART = PIPE.run(src_a, src_b, scope=CFG.scope or None)
print("comparison status:", RES["comparison_status"], "| decisions:", RES["summary_counts"],
      "| latency:", RES["efficiency"]["latency_s"], "s | GPU:", RES["efficiency"]["gpu"])
display(pd.DataFrame(RES["component_status"])[["component", "state", "detail"]])

## 8. Inspect the result
### 8.1 Report (verified claims only)

In [ ]:
from IPython.display import Markdown
print("verbalizer:", RES["report"]["status"], RES["report"].get("problems", ""))
display(Markdown(RES["report"]["text"]))

### 8.2 Pair validation and image quality

In [ ]:
V = RES["validation"]
print("status:", V["status"], "| warnings:", V["warnings"], "| DINOv2 compat:", V["compat_similarity"])
display(pd.DataFrame({"A": V["image_a"]["quality"], "B": V["image_b"]["quality"]}))

### 8.3 Correspondence and alignment diagnostics
The aligned view is **additional evidence only**; the originals are never modified.

In [ ]:
from idu.report import figure_overview, figure_matches, figure_maps
AL_ = {k: v for k, v in RES["alignment"].items() if k not in ("M_ba",)}
display(pd.DataFrame([AL_]).T.rename(columns={0: "value"}))
print("M_ba (B->A):\n", pd.DataFrame(RES["alignment"]["M_ba"]).round(4))
figure_matches(ART);

### 8.4 Low-level evidence maps
Values are robust z-scores (median/MAD units, contrast-conditional), **not probabilities**. `z_unaligned` shows the residual before alignment, and `flow` / `uncertainty` show local misregistration risk.

In [ ]:
if ART.get("low"):
    figure_maps(ART["low"], CFG.z_high)
    print("photometric gains (B->A):", RES["lowlevel"]["photometric_gains"], "| params:", RES["lowlevel"]["params"])

### 8.5 Candidate regions (low-level) with nuisance signatures and native-resolution re-inspection

In [ ]:
if RES["lowlevel"]:
    rows = [{"obs": r["obs_id"], "bbox_a": [round(v) for v in r["bbox_a"]], "area_px": r["stats"]["area_px"],
             "peak_z": round(r["stats"]["peak_z"], 1), "dominant": r["dominant_map"], "nuisance": r["nuisance"],
             "native_confirmed": r["native"].get("confirmed"), "native_peak_z": r["native"].get("peak_z_native")}
            for r in RES["lowlevel"]["regions"]]
    display(pd.DataFrame(rows))

### 8.6 OCR evidence

In [ ]:
O = RES.get("ocr")
if O:
    display(pd.DataFrame([{"before": d["before"], "after": d["after"], "edits": [(e["op"], e["a_str"], e["b_str"]) for e in d["edits"]],
                           "conf": (round(d["conf_a"], 2), round(d["conf_b"], 2)), "reread": d.get("reread_status"),
                           "pixel_verdict": d["visual"]["verdict"], "diff_in_span": d["visual"].get("diff_frac_in_edit_span")} for d in O["diffs"]]))
    print("added:", [(x["text"], x["status"]) for x in O["added"]], "| removed:", [(x["text"], x["status"]) for x in O["removed"]])
else:
    print("OCR did not run (see component status).")

### 8.7 Semantic hypotheses (unverified until §8.8)

In [ ]:
S = RES["semantic"]
if S.get("vlm"):
    print("VLM parsed JSON:", S["vlm"]["parsed"], "| model-reported nuisance:", S["vlm"]["nuisance"])
    display(pd.DataFrame(S["vlm"]["hypotheses"]))
if S.get("m3diff"):
    print("M3Diff caption:", S["m3diff"]["raw"]); display(pd.DataFrame(S["m3diff"]["hypotheses"]))

### 8.8 Evidence graph and claim decisions
Each claim lists its supporting and contradicting observations, its independent evidence families, the per-dimension confidence, and the explicit reason for the decision.

In [ ]:
CL = pd.DataFrame([{"claim": c["claim_id"], "decision": c["decision"], "type": c["change_type"], "object": c["object_or_region"],
                    "before": c["before"], "after": c["after"], "bbox_a": c["bbox_a"] and [round(v) for v in c["bbox_a"]],
                    "bbox_b": c["bbox_b"] and [round(v) for v in c["bbox_b"]], "loc": c["location_semantics"],
                    "families": c["confidence"].get("claim", {}).get("independent_support_families"),
                    "support": c["supporting"], "contra": c["contradicting"], "nuisance": c["nuisance_explanations"],
                    "raw_score(heuristic)": c["raw_support_score"], "calibrated": c["calibrated_confidence"],
                    "why": "; ".join(c["decision_reasons"])} for c in RES["claims"]])
display(CL)
OBS = pd.DataFrame(RES["evidence_graph"]["observations"])
display(OBS[["observation_id", "source", "observation_type", "family", "image", "bbox_a", "raw_score", "score_kind", "supports", "contradicts", "depends_on"]])

In [ ]:
# drill into one claim: its full evidence chain
CLAIM_ID = RES["claims"][0]["claim_id"] if RES["claims"] else None
if CLAIM_ID:
    c = next(x for x in RES["claims"] if x["claim_id"] == CLAIM_ID)
    print(json.dumps({k: c[k] for k in ("claim_id", "decision", "decision_reasons", "confidence", "details")}, indent=1, default=str)[:4000])
    for oid in c["supporting"] + c["contradicting"]:
        o = next(x for x in RES["evidence_graph"]["observations"] if x["observation_id"] == oid)
        print(("SUPPORTS   " if oid in c["supporting"] else "CONTRADICTS"), oid, o["source"], o["observation_type"], str(o["raw"])[:200])

### 8.9 Overlays (green = accepted, orange = needs review, grey = abstained)

In [ ]:
figure_overview(ART, ART["claims"]);
if DEMO_GT: print("demo ground truth:", DEMO_GT)

### 8.10 Machine-readable result

In [ ]:
from idu.schema import to_jsonable
print(json.dumps(to_jsonable({k: RES[k] for k in ("run_id", "comparison_status", "summary_counts", "claims")}), indent=1)[:6000])

## 9. Export
`result.json` (full record: config, environment, validation, alignment, observations, claims, report, timings), `report.md`, overlays, every evidence map, per-claim masks, and figures are already written under `outputs/<run_id>/`. This cell zips them.

In [ ]:
import shutil
print("artifacts:", RES.get("artifacts_dir"))
json.dump({**ENV, "libraries": libs}, open(Path(RES["artifacts_dir"]) / "environment_report.json", "w"), indent=2)
zip_path = shutil.make_archive(RES["artifacts_dir"], "zip", RES["artifacts_dir"])
print("zip:", zip_path)
if IN_COLAB:
    from google.colab import files; files.download(zip_path)

## 10. Evaluation and ablations
The synthetic pairs have **exact ground truth** (text edits, colour changes, add/remove, scratches, plus nuisance-only lighting, JPEG, camera shift and noise). Use them for regression testing and calibration plumbing, **not** for research claims. For research claims, evaluate on an annotated set from your domain in the same `pairs.jsonl` format (see `idu/evaluate.py`).

In [ ]:
from idu.evaluate import make_synthetic_suite, load_pairs, evaluate, run_ablations, calibration_report
DATASET = ""          # path to your own pairs.jsonl; empty -> synthetic suite
N_PER_CATEGORY = 2
pairs = load_pairs(DATASET) if DATASET else load_pairs(make_synthetic_suite("outputs/synthetic_suite", n_per_category=N_PER_CATEGORY))
EV = evaluate(PIPE, pairs, include_review=False)
display(pd.DataFrame([EV["aggregate"]]).T.rename(columns={0: "accepted-only"}))
display(pd.DataFrame(EV["per_category"]).T[["n", "tp", "fp", "fn", "precision", "recall", "false_change_rate_nuisance_pairs", "mean_box_iou", "text_cer"]])

In [ ]:
import matplotlib.pyplot as plt
rc = EV["risk_coverage"]
if rc["coverage"]:
    plt.figure(figsize=(5, 4)); plt.plot(rc["coverage"], rc["risk"]); plt.xlabel("coverage"); plt.ylabel("risk (error rate)")
    plt.title(f"risk-coverage of raw_support_score, AURC={rc['aurc']:.3f}"); plt.grid(alpha=.3); plt.show()
print(calibration_report(EV["calibration_samples"]))

In [ ]:
# Ablations (Section 9). Each variant re-runs the whole set; budget time accordingly on GPU.
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    TAB = run_ablations(PIPE, pairs)
    display(pd.DataFrame(TAB)[["variant", "precision", "recall", "f1", "missed_change_rate", "unsupported_claim_rate",
                               "false_change_rate_nuisance_pairs", "mean_box_iou", "text_cer", "review_rate", "aurc", "mean_latency_s", "max_peak_gpu_gb"]])

## 11. Calibration (held-out labelled data only)
This fits a calibrator that maps the heuristic `raw_support_score` to P(claim correct). The fit is refused with fewer than 50 labelled claims or when only one class is present. **Fit on one split and evaluate on another.** Without a calibrator, `calibrated_confidence` stays `null` in every report.

In [ ]:
from idu.calibration import Calibrator, ece
FIT_CALIBRATOR = False
if FIT_CALIBRATOR:
    import random
    s = EV["calibration_samples"][:]; random.Random(0).shuffle(s)
    fit, held = s[: len(s) // 2], s[len(s) // 2:]
    cal = Calibrator("platt").fit([x["score"] for x in fit], [x["correct"] for x in fit])
    p_held = cal.predict([x["score"] for x in held])
    print("held-out ECE raw:", ece([x["score"] for x in held], [x["correct"] for x in held])[0], "| calibrated:", ece(p_held, [x["correct"] for x in held])[0])
    cal.save("outputs/calibrator.json")
    # to use it: set CFG.calibrator_path = "outputs/calibrator.json" (it applies to subsequent runs)

## 12. Troubleshooting and limitations

| Symptom | Cause | Fix |
|---|---|---|
| `CUDA out of memory` while loading the VLM | another heavy model is resident, or an 8B model is in bf16 | Heavy models are swapped automatically. Use Qwen3-VL-**4B**, or set `vlm_4bit=True`, or lower `vlm_max_image_pixels` |
| `transformers ... lacks Qwen3-VL` | transformers < 4.57 | `pip install -U "transformers>=4.57.1,<5"` |
| LightGlue `failed` in self-test | package missing or GitHub blocked | `pip install git+https://github.com/cvg/LightGlue.git`. The pipeline falls back to LoFTR, then ORB, and **records the substitution** |
| SuperPoint licensing concern | weights are non-commercial | `lightglue_features="disk"` or `"aliked"` |
| Alignment `unreliable` on documents/screenshots | few features | set `domain="document"`/`"screenshot"` (identity is tried first). For same-size captures, identity is assumed only when global NCC > 0.8 |
| Many `needs_review` on JPEG inputs | compression drift is visually real | expected for pixel-only evidence. Enable the VLM (crop re-inspection can explain nuisance) or provide higher-quality inputs |
| OCR misses small text | text below the detector scale | raise `ocr_upscale` and check `ocr_langs`. Claims for text under `ocr_min_char_height_px` are never stated exactly |
| M3Diff `failed` | env path, repo or weights | run `scripts/setup_m3diff_env.sh` (Linux/WSL), then set `m3diff_python`; check its stderr in the component table |
| AnyChange `failed` | `torchange` or SAM checkpoint missing | `pip install torchange`; download `sam_vit_h_4b8939.pth` to `checkpoints/` |
| Everything abstains | evidence genuinely insufficient (poor alignment / quality) | read `decision_reasons`; abstention is a valid output |

**Known limitations (honest):**
- The thresholds (`z_high`, `z_low`, noise floors, pixel-only acceptance bar, text pixel-verdict cut-offs) are hand-set defaults. They were checked only on synthetic pairs and must be tuned on labelled data from your domain (§10).
- `raw_support_score` is a heuristic ranking score, not a probability. Calibrated confidence exists only after §11.
- Character boxes inside a text line are proportional estimates. EasyOCR gives line-level confidence only.
- Global geometric models are approximate for 3D scenes with parallax. The local uncertainty and flow signatures flag this, but they do not repair it.
- VLM and M3Diff outputs are hypotheses. Their boxes are hints that are re-checked against pixels. M3Diff in 4-bit may differ from the paper's numbers.
- AnyChange was designed for remote sensing, so its masks are treated as `containing_object` evidence.
- An empty difference map is not proof that nothing changed. The report says which branches ran.

## 13. Cleanup
This releases GPU memory held by cached models. Model caches on disk are **never** deleted automatically.

In [ ]:
try:
    PIPE.close(); print("released")
except NameError:
    pass